# What does a NIM-style service give you over traditional serving? One client, three backends

Question: what do you gain from a NIM-style service over traditional model serving, can one client move between self-hosted and cloud without code changes, and how do you keep private data off the cloud?

Setup: Kaggle with two Tesla T4 GPUs (16 GB each, Turing, no bf16, so fp16) and internet on. A real NVIDIA NIM container needs Docker and an NGC login. Kaggle has neither. So **vLLM stands in for a self-hosted NIM**. vLLM is not a NIM. It is an inference engine with an OpenAI-compatible API. A NIM packages an engine, CUDA libraries and that kind of API in one container, and NVIDIA supports it. This notebook tests the API and the serving behavior. It does not test the packaging.

| Backend | What it is | Where it runs | Model |
|---|---|---|---|
| A | Traditional serving: a small Python HTTP server around Transformers `generate`, with its own JSON format | GPU 0, own process | `Qwen/Qwen2.5-1.5B-Instruct` |
| B | NIM-style serving: vLLM with the OpenAI API (stand-in for a self-hosted NIM) | GPU 1 | the same Qwen model |
| C | Hosted NIM: the NVIDIA API catalog at `integrate.api.nvidia.com/v1` | NVIDIA cloud | `nvidia/nemotron-3.5-bolt-e4b`, or `google/gemma-3-4b-it` if the first is not listed |

C needs an `NVIDIA_API_KEY` secret. Without it, the hosted parts print a note and skip. The local parts still run.

Terms used below:

- **Traditional model serving**: you write the server yourself. It has your own request format, and it has no standard API, health, metrics or streaming unless you add them.
- **NIM** (NVIDIA Inference Microservice): a packaged model server with an OpenAI-compatible API.
- **Self-hosted**: the server runs on hardware you control. Your data does not leave it.
- **Hosted**: NVIDIA runs the server. Your data goes to NVIDIA over the internet.
- **OpenAI-compatible API**: the `/v1/chat/completions` request and reply format. Many clients already speak it.
- **Throughput**: output tokens per second, summed over all requests.
- **TTFT** (time to first token): the wait until the first output token arrives. It needs streaming.
- **p50 / p95 latency**: 50 (or 95) out of 100 requests finish at or below this time.
- **Continuous batching**: the server adds new requests to a running batch at every step. A server that handles one request at a time does not do this.
- **Conformance check**: the same test sent to two backends. A pass means both answer in the same format.
- **PII** (personally identifiable information): data that names or reaches a person, such as an email, phone number, bank number or home address.
- **Fail closed**: when a safety step breaks, the request stops. It does not continue by an unsafe path.
- **Guardrail model**: a small model that checks a request or a reply against a policy.
- **Embedding**: a vector of numbers for a text. Texts with a similar meaning get vectors that point the same way.
- **RAG** (retrieval-augmented generation): find the best documents, then give them to the LLM as sources.
- **Blueprint**: an NVIDIA reference architecture built from NIM parts, with reference code and deployment files.
- **AI Factory layers**: infrastructure, platform, models and applications.
- **Workload profile**: the shape of the load. Examples are short and fast, or long and heavy on decode.
- **KV cache**: the memory that holds the attention keys and values of earlier tokens.
- **AWQ**: a 4-bit weight format that makes a model about 2.7 times smaller. **Tensor parallelism**: one model split over several GPUs.

## Predict before running

Write your own guess first. Then compare it with the tables.

| Question | What you measure | Your guess |
|---|---|---|
| Which backend gives more tokens per second at concurrency 8 and 16, A or B? Why? | Throughput table |  |
| What happens to the p95 latency of A when concurrency grows? | p95 latency of A |  |
| How many client code lines does A need? How many change to move from B to C? | Client line count |  |
| What comes built in on B that you must write yourself on A? | Endpoint probe |  |
| Which conformance checks fail on B? Which fail on C? | Conformance table |  |
| Does the regex detector catch every PII prompt? How many PII prompts reach the cloud? | Leak count, precision, recall |  |
| Does the egress check stop a PII prompt that the router sent to the cloud by mistake? | Egress test |  |
| When B stops, where do PII requests go? When C breaks, where do other requests go? | Failover tables |  |
| Does the pipeline still run after you change only the embedding model name? | Blueprint swap table |  |
| Which profile does the picker choose for a 7B model on one T4? On two T4s? | Profile table |  |

In [ ]:
# 1. Install vLLM (into the Kaggle VM, not your laptop). It also installs torch and transformers for backend A.
!pip install -q vllm 2>&1 | tail -3
!python -c "import vllm; print('vllm', vllm.__version__)"
!nvidia-smi --query-gpu=index,name,memory.total,compute_cap --format=csv

In [ ]:
# 2. Helpers: config, key, one HTTP function (timeout, retry), one chat client for every OpenAI-style backend
# Do not import torch with CUDA in this kernel. Backend A runs in its own process.
import collections, http.client, inspect, json, math, os, re, statistics, subprocess, sys, time, urllib.error, urllib.request
from concurrent.futures import ThreadPoolExecutor
import pandas as pd

LOCAL_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
CLOUD_BASE = os.environ.get("NIM_BASE_URL", "https://integrate.api.nvidia.com/v1")
CLOUD_PREFERRED, CLOUD_FALLBACK = "nvidia/nemotron-3.5-bolt-e4b", "google/gemma-3-4b-it"
PORT_A, PORT_B = 8101, 8102          # not 8000: other notebooks use it
N_TOKENS = 96                         # output tokens per request in the load test
REQ_TIMEOUT_S, MAX_RETRIES, BACKOFF_S, RETRY_CAP_S = 90, 4, 2.0, 30.0
RUN_DEADLINE = time.time() + 28 * 60  # every cloud call stops here
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
Backend = collections.namedtuple("Backend", "name base model key")
NAN, SUMMARY = float("nan"), []       # SUMMARY rows feed the final table

def note(section, metric, value): SUMMARY.append(dict(section=section, metric=metric, value=value))
def dl(seconds=120): return min(RUN_DEADLINE, time.time() + seconds)
def pct(xs, q): xs = sorted(xs); return xs[min(len(xs) - 1, int(q * len(xs)))] if xs else NAN
def src_lines(fn):
    try: return len(inspect.getsource(fn).splitlines())
    except (OSError, TypeError): return "n/a"   # source not available in this environment

def get_key():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("NVIDIA_API_KEY")
    except Exception:
        return os.environ.get("NVIDIA_API_KEY") or None   # None: the hosted parts are skipped

def api(be, path, body=None, timeout=REQ_TIMEOUT_S, retries=MAX_RETRIES, deadline=None, stream=False):
    # GET without body, POST with a JSON body. Retries 429, 5xx and network errors (Retry-After honoured).
    out = dict(ok=False, status=0, json=None, text="", error="", retries=0, resp=None)
    data = None if body is None else json.dumps(body).encode()
    headers = {"Content-Type": "application/json"}
    if be.key: headers["Authorization"] = "Bearer " + be.key   # the key is never printed
    for attempt in range(retries + 1):
        left = (deadline - time.time()) if deadline else timeout
        if left <= 0:
            out["error"] = "deadline"; break
        wait = None
        try:
            r = urllib.request.urlopen(urllib.request.Request(be.base + path, data, headers), timeout=min(timeout, left))
            if stream:
                out.update(ok=True, status=r.status, resp=r); return out
            with r: raw = r.read()
            out.update(status=r.status, text=raw.decode("utf-8", "replace"))
            out["json"] = json.loads(out["text"])
            out.update(ok=True, error=""); return out
        except urllib.error.HTTPError as e:           # 429 and 5xx are retried, other codes are final
            try:
                out["text"] = e.read().decode("utf-8", "replace"); out["json"] = json.loads(out["text"])
            except (OSError, ValueError, http.client.HTTPException): pass
            out.update(status=e.code, error=f"HTTP {e.code}")
            if e.code != 429 and e.code < 500: return out
            try: wait = float(e.headers.get("Retry-After", ""))
            except ValueError: wait = None
        except ValueError:                            # status 200 but the body is not JSON
            out["error"] = "bad json"; return out
        except (OSError, http.client.HTTPException) as e:   # timeout, refused, reset, cut-off body
            out["error"] = type(getattr(e, "reason", e)).__name__
        if attempt == retries: break
        wait = min(BACKOFF_S * 2 ** attempt if wait is None else wait, RETRY_CAP_S)
        if deadline and time.time() + wait >= deadline: break
        time.sleep(wait); out["retries"] += 1
    return out

def chat(be, prompt, max_tokens=N_TOKENS, stream=False, stop=None, extra=None, **kw):
    # THE client for B and C. Only `be` (base URL, model, key) changes between backends. kw goes to api().
    messages = [{"role": "user", "content": prompt}] if isinstance(prompt, str) else prompt
    body = {"model": be.model, "messages": messages, "max_tokens": max_tokens, "temperature": 0, **(extra or {})}
    if stop: body["stop"] = stop
    if stream: body.update(stream=True, stream_options={"include_usage": True})
    out = dict(ok=False, content="", tokens=0, ttft=NAN, latency=NAN, finish="", error="", status=0, msg={}, chunks=0, done=False, retries=0)
    t0 = time.perf_counter()
    h = api(be, "/chat/completions", body, stream=stream, **kw)
    out.update(error=h["error"], status=h["status"], retries=h["retries"])
    try:
        if h["ok"] and not stream:
            ch = h["json"]["choices"][0]
            out.update(ok=True, msg=ch["message"], content=ch["message"].get("content") or "", finish=ch.get("finish_reason") or "",
                       tokens=(h["json"].get("usage") or {}).get("completion_tokens") or 0)
        elif h["ok"]:
            parts = []
            with h["resp"] as r:
                for raw in r:
                    line = raw.decode("utf-8", "replace").strip()
                    if not line.startswith("data:"): continue
                    if line[5:].strip() == "[DONE]":
                        out["done"] = True; break
                    ev = json.loads(line[5:])
                    if ev.get("usage"): out["tokens"] = ev["usage"].get("completion_tokens") or 0
                    for ch in ev.get("choices") or []:
                        d = ch.get("delta") or {}
                        if ch.get("finish_reason"): out["finish"] = ch["finish_reason"]
                        if d.get("content") or d.get("reasoning_content") or d.get("reasoning"):
                            out["chunks"] += 1
                            if out["ttft"] != out["ttft"]: out["ttft"] = time.perf_counter() - t0
                            if d.get("content"): parts.append(d["content"])
                    if kw.get("deadline") and time.time() > kw["deadline"]: raise TimeoutError("deadline")
            out.update(ok=True, content="".join(parts), tokens=out["tokens"] or out["chunks"])
    except (ValueError, KeyError, IndexError, TypeError, AttributeError, OSError, http.client.HTTPException) as e:
        out.update(ok=False, error="bad reply " + type(e).__name__)
    out["latency"] = time.perf_counter() - t0
    return out

def generate_a(be, text, n_tokens, **kw):
    # The client for backend A: its own path, its own request and reply format. No streaming.
    t0 = time.perf_counter()
    h = api(be, "/generate", {"text": text, "n_tokens": n_tokens}, **kw)
    j = h["json"] if isinstance(h["json"], dict) else {}
    return dict(ok=h["ok"] and "output" in j, tokens=j.get("tokens", 0), ttft=NAN, error=h["error"], latency=time.perf_counter() - t0)

def bench(call, prompts, conc, n):
    # n requests, `conc` at a time. call(prompt) returns a dict with ok, tokens, latency, ttft.
    t0 = time.perf_counter()
    with ThreadPoolExecutor(conc) as ex: rs = list(ex.map(call, [prompts[i % len(prompts)] for i in range(n)]))
    wall, ok = time.perf_counter() - t0, [r for r in rs if r["ok"]]
    ttft = [r["ttft"] for r in ok if r["ttft"] == r["ttft"]]
    return dict(concurrency=conc, requests=n, errors=n - len(ok), tokens_per_req=sum(r["tokens"] for r in ok) / max(1, len(ok)),
                tok_s=sum(r["tokens"] for r in ok) / wall, p50_s=pct([r["latency"] for r in ok], 0.5),
                p95_s=pct([r["latency"] for r in ok], 0.95), ttft_p50_s=pct(ttft, 0.5))

In [ ]:
# 3. Helpers: backend A server script, server start and stop (same pattern as the vLLM notebooks)
A_SCRIPT = r'''
import json, sys, threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
MODEL, PORT = sys.argv[1], int(sys.argv[2])
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16).to("cuda").eval()
LOCK = threading.Lock()   # one request at a time: no batching, no scheduler

class H(BaseHTTPRequestHandler):
    def send(self, code, obj):
        body = json.dumps(obj).encode()
        self.send_response(code); self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body))); self.end_headers(); self.wfile.write(body)
    def do_GET(self):
        self.send(200, {"status": "ok"}) if self.path == "/health" else self.send(404, {"detail": "not found"})
    def do_POST(self):
        if self.path != "/generate": return self.send(404, {"detail": "not found"})
        try:
            req = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
            n = int(req.get("n_tokens", 64))
            text = tok.apply_chat_template([{"role": "user", "content": req["text"]}], tokenize=False, add_generation_prompt=True)
        except (ValueError, KeyError, TypeError):
            return self.send(400, {"detail": "send {text, n_tokens}"})
        with LOCK:
            ids = tok(text, return_tensors="pt").to("cuda")
            with torch.no_grad():
                out = model.generate(**ids, max_new_tokens=n, min_new_tokens=n, do_sample=False)   # min = max: fixed output length
        new = out[0, ids["input_ids"].shape[1]:]
        self.send(200, {"output": tok.decode(new, skip_special_tokens=True), "tokens": int(new.shape[0])})
    def log_message(self, *a): pass

ThreadingHTTPServer(("127.0.0.1", PORT), H).serve_forever()
'''

def gpu_used_mib():
    out = subprocess.check_output(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], text=True)
    return [int(x) for x in out.split()]

def launch(cmd, gpu, log):
    srv = subprocess.Popen(cmd, stdout=open(log, "w"), stderr=subprocess.STDOUT, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)))
    srv.log = log
    return srv

def wait_ready(srv, url, limit_s=1200):
    t0 = time.time()
    while True:
        try:
            urllib.request.urlopen(url, timeout=5).close(); return time.time() - t0
        except Exception:
            if srv.poll() is not None: raise RuntimeError("server exited, see " + srv.log)
            if time.time() - t0 > limit_s:
                srv.kill(); raise RuntimeError("server not ready after 20 min")
            time.sleep(2)

def stop_server(srv, gpu):
    srv.terminate()
    try: srv.wait(timeout=90)
    except subprocess.TimeoutExpired: srv.kill(); srv.wait()
    for _ in range(60):   # wait until the GPU memory is free again
        if gpu_used_mib()[gpu] < 500: return
        time.sleep(2)
    print("warning: GPU memory not free after 120 s:", gpu_used_mib())

In [ ]:
# 4. Start backend A (GPU 0) and backend B (GPU 1). Both load at the same time.
open(f"{OUT_DIR}/backend_a.py", "w").write(A_SCRIPT)
VLLM_CMD = ["vllm", "serve", LOCAL_MODEL, "--dtype", "half", "--max-model-len", "4096", "--gpu-memory-utilization", "0.9",
            "--port", str(PORT_B), "--enable-auto-tool-choice", "--tool-call-parser", "hermes"]   # the tool flags: a NIM ships them set
PROCS = {"A": launch([sys.executable, f"{OUT_DIR}/backend_a.py", LOCAL_MODEL, str(PORT_A)], 0, f"{OUT_DIR}/backend_a.log"),
         "B": launch(VLLM_CMD, 1, f"{OUT_DIR}/backend_b.log")}
for k, url in (("A", f"http://127.0.0.1:{PORT_A}/health"), ("B", f"http://127.0.0.1:{PORT_B}/health")):
    print(f"backend {k} ready after {wait_ready(PROCS[k], url):.0f} s")
A = Backend("A traditional", f"http://127.0.0.1:{PORT_A}", LOCAL_MODEL, None)
B = Backend("B NIM-style (vLLM)", f"http://127.0.0.1:{PORT_B}/v1", LOCAL_MODEL, None)

In [ ]:
# 5. Backend C: key, hosted model list (GET /models needs no key), model choice
KEY = get_key()
_h = api(Backend("cloud", CLOUD_BASE, "", None), "/models", timeout=30, retries=2)
HOSTED = {m.get("id") for m in (_h["json"] or {}).get("data", [])} if _h["ok"] and isinstance(_h["json"], dict) else set()
CLOUD_MODEL = CLOUD_PREFERRED if (CLOUD_PREFERRED in HOSTED or not HOSTED or CLOUD_FALLBACK not in HOSTED) else CLOUD_FALLBACK
C = Backend("C hosted NIM", CLOUD_BASE, CLOUD_MODEL, KEY)
HAVE_C = bool(KEY)
print("hosted models listed:", len(HOSTED) or "listing failed", "| C model:", CLOUD_MODEL, "| key found:", HAVE_C)   # never print the key
if not HAVE_C:
    print("NOTE: no NVIDIA_API_KEY. Add-ons > Secrets > add NVIDIA_API_KEY, then switch it on for this notebook. Hosted parts will be skipped.")

## Experiment 1: traditional serving (A) against NIM-style serving (B)

Same GPU type (T4), same model, same prompts, same output length (96 tokens, fixed on both backends). The test sends 4, 16 and 32 requests at concurrency 1, 8 and 16. A runs one request at a time. B uses continuous batching. B streams, so its TTFT is measured. A cannot stream, so its TTFT is `NaN`. One warm-up request runs first on each backend.

The comparison is fair for speed: the two backends differ in the serving software, not in the model or the GPU type. It is not a claim about what a packaged NIM would do on a bigger GPU.

In [ ]:
# 6. Experiment 1a: load test on A and B
LEVELS = {1: 4, 8: 16, 16: 32}   # concurrency -> number of requests
TOPICS = ["how a refrigerator works", "why the sea is salty", "what a hash table is", "how vaccines train the body",
          "how a bicycle gear works", "what inflation is", "how GPS finds a position", "why bread rises",
          "what a black hole is", "how email travels", "what a database index does", "how a rainbow forms",
          "what photosynthesis is", "how a bridge carries weight", "what encryption does", "how a battery stores energy"]
PROMPTS = [f"Explain {t} in plain words, with one example." for t in TOPICS]
CALLS = {"A traditional": lambda p: generate_a(A, p, N_TOKENS, timeout=300),
         "B NIM-style": lambda p: chat(B, p, N_TOKENS, stream=True, extra={"ignore_eos": True})}   # ignore_eos: a vLLM extra, fixes the length
rows = []
for name, call in CALLS.items():
    call(PROMPTS[0])   # warm-up
    for c, n in LEVELS.items():
        rows.append(dict(backend=name, **bench(call, PROMPTS, c, n)))
BENCH = pd.DataFrame(rows)
BENCH.round(2).to_csv(f"{OUT_DIR}/nim3_serving_bench.csv", index=False)
print(BENCH.round(2).to_string(index=False))
for c in LEVELS:
    a, b = BENCH[(BENCH.backend == "A traditional") & (BENCH.concurrency == c)].iloc[0], BENCH[(BENCH.backend == "B NIM-style") & (BENCH.concurrency == c)].iloc[0]
    note("1 serving", f"tok/s at concurrency {c}: A / B", f"{a.tok_s:.1f} / {b.tok_s:.1f}")
    note("1 serving", f"p95 latency s at concurrency {c}: A / B", f"{a.p95_s:.1f} / {b.p95_s:.1f}")
    note("1 serving", f"B TTFT p50 s at concurrency {c}", f"{b.ttft_p50_s:.2f}")

In [ ]:
# 7. Experiment 1b: what comes built in? Probe each endpoint on A and B, and count client lines
def has(be, path, post=False):   # True when the route exists: any status except 404 (a 400 means the route is there)
    root = be._replace(base=be.base.removesuffix("/v1"))
    return api(root, path, {} if post else None, retries=0, timeout=10)["status"] not in (0, 404)

b_streams = bool((BENCH[BENCH.backend == "B NIM-style"].ttft_p50_s > 0).any())
PROBE = pd.DataFrame([
    dict(feature="health endpoint", A=has(A, "/health"), B=has(B, "/health")),
    dict(feature="metrics endpoint (Prometheus)", A=has(A, "/metrics"), B=has(B, "/metrics")),
    dict(feature="model list (GET /v1/models)", A=has(A, "/v1/models"), B=has(B, "/v1/models")),
    dict(feature="OpenAI chat route", A=has(A, "/v1/chat/completions", True), B=has(B, "/v1/chat/completions", True)),
    dict(feature="token streaming (measured)", A=False, B=b_streams),
])
print(PROBE.to_string(index=False))
n_a, n_chat = src_lines(generate_a), src_lines(chat)
differ = [f for f in ("base", "model", "key") if getattr(B, f) != getattr(C, f)]
print(f"\nclient code for A (own format, no streaming): {n_a} lines. A client for A cannot talk to B or C.")
print(f"client code for B and C: the same chat() function ({n_chat} lines). Moving B -> C changes 0 code lines and these values: {differ}")
PROBE.to_csv(f"{OUT_DIR}/nim3_builtin_features.csv", index=False)
for _, r in PROBE.iterrows(): note("1 serving", "built in: " + r.feature + " (A / B)", f"{r.A} / {r.B}")
note("1 serving", "client lines: A function / shared B+C function / code lines changed B to C", f"{n_a} / {n_chat} / 0")

## Experiment 2: one client, two OpenAI-style backends (B and C)

The same `chat()` function runs the same eight checks on B (self-hosted stand-in) and C (hosted NIM). Only the `Backend` values change: base URL, model and key. A check passes when the reply has the shape the OpenAI API defines. The test does not judge the quality of the answers.

Do not compare latency between B and C. They run a different model on different hardware, and C adds the internet. The table has no latency column on purpose.

In [ ]:
# 8. Helpers: conformance checks. Each check returns (passed, short detail).
TOOLS = [{"type": "function", "function": {"name": "get_weather", "description": "Get the weather for a city",
          "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}}]

def chk_models(be):
    h = api(be, "/models", timeout=30, retries=1, deadline=dl(60))
    ids = [m.get("id") for m in h["json"].get("data", [])] if h["ok"] and isinstance(h["json"], dict) else []
    return be.model in ids, f"{len(ids)} models listed"
def chk_chat(be):
    r = chat(be, "Reply with the single word: ok", 512, deadline=dl())
    return r["ok"] and bool(r["content"].strip()), f"{r['tokens']} tokens, finish={r['finish'] or r['error']}"
def chk_stream(be):
    r = chat(be, "Count from 1 to 20, separated by commas.", 512, stream=True, deadline=dl())   # a one-word reply fits in one chunk
    return r["ok"] and r["chunks"] > 1 and r["done"] and bool(r["content"].strip()), f"{r['chunks']} chunks, done={r['done']}"
def chk_max_tokens(be):
    r = chat(be, "Count from 1 to 200, one number per line.", 8, deadline=dl())
    return r["ok"] and 0 < r["tokens"] <= 8 and r["finish"] == "length", f"{r['tokens']} tokens, finish={r['finish'] or r['error']}"
def chk_stop(be):
    r = chat(be, "Count from 1 to 10 as digits separated by commas. Write nothing else.", 512, stop=["5"], deadline=dl())
    return r["ok"] and "5" not in r["content"] and r["finish"] == "stop", f"finish={r['finish'] or r['error']}, text={r['content'][:20]!r}"
def chk_json(be):
    r = chat(be, 'Reply with a JSON object with the key "answer" and the value 4.', 512, extra={"response_format": {"type": "json_object"}}, deadline=dl())
    try: ok = r["ok"] and isinstance(json.loads(r["content"]), dict)
    except ValueError: ok = False
    return ok, f"text={r['content'][:30]!r}" if r["ok"] else r["error"]
def chk_tool(be):
    r = chat(be, "What is the weather in Lyon right now? Use the tool.", 512, extra={"tools": TOOLS, "tool_choice": "auto"}, deadline=dl())
    calls = r["msg"].get("tool_calls") or []
    try: ok = r["ok"] and calls[0]["function"]["name"] == "get_weather" and isinstance(json.loads(calls[0]["function"]["arguments"]), dict)
    except (IndexError, KeyError, TypeError, ValueError): ok = False
    return ok, f"{len(calls)} tool call(s)" if r["ok"] else r["error"]
def chk_error(be):
    h = api(be, "/chat/completions", {"model": "no-such-model-xyz", "messages": [{"role": "user", "content": "hi"}], "max_tokens": 4},
             retries=0, timeout=30, deadline=dl(60))
    j = h["json"] if isinstance(h["json"], dict) else {}
    return 400 <= h["status"] < 500 and "error" in j, f"HTTP {h['status']}, top-level keys {sorted(j)[:4]}"

CHECKS = [("list models", chk_models), ("chat", chk_chat), ("streaming", chk_stream), ("max_tokens respected", chk_max_tokens),
          ("stop sequence", chk_stop), ("JSON reply", chk_json), ("tool call", chk_tool), ("error format, bad model name", chk_error)]

def run_conformance(be):
    rows = []
    for name, fn in CHECKS:
        try: ok, detail = fn(be)
        except Exception as e: ok, detail = False, "check crashed: " + type(e).__name__
        rows.append((name, "pass" if ok else "FAIL", detail))
    return rows

In [ ]:
# 9. Experiment 2: conformance table, B against C
RES = {"B": run_conformance(B)}
if HAVE_C: RES["C"] = run_conformance(C)
else: print("NOTE: C skipped, no key. Only B is checked.")
CONF = pd.DataFrame({"check": [n for n, _ in CHECKS]})
for k, rows_ in RES.items():
    CONF[k] = [r[1] for r in rows_]
    CONF[k + " detail"] = [r[2] for r in rows_]
CONF.to_csv(f"{OUT_DIR}/nim3_conformance.csv", index=False)
print(CONF.to_string(index=False))
for k, rows_ in RES.items(): note("2 conformance", f"checks passed on {k}", f"{sum(r[1] == 'pass' for r in rows_)} of {len(rows_)}")

## Experiment 3: hybrid routing for data control

Rule: a prompt that holds PII goes to self-hosted B. Every other prompt goes to hosted C. A local detector decides. It uses regular expressions for emails, phone numbers, IBAN-like numbers and card-like numbers. All test data is fake (`example.com`, `555` numbers, `TEST` bank codes, public test card numbers).

The test set has 20 prompts with known labels. 10 hold PII. Two of those hold only a name and a street address, which the regular expressions cannot see. One non-PII prompt holds a long build number that looks like a card number.

Three safety layers sit in the code:

1. The router sends detected PII to B only.
2. If B is down, the request fails. It never goes to C (**fail closed**).
3. An **egress check** sits in the only function that sends to C. It scans the text again and refuses to send PII. The function writes each attempt to a ledger before the request leaves.

A **leak** is a prompt that truly holds PII (by the label) and appears in the ledger as sent to C. The ledger lives in the client, so it does not depend on what C reports.

In [ ]:
# 10. Helpers: PII detector, test prompts with labels, router, egress check
PII_PATTERNS = {
    "email": r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+",
    "phone": r"(?<![\w.])(?:\+\d{1,3}[ .-]?)?(?:\d{3}[ .-]){1,2}\d{4}(?!\d)",
    "iban": r"\b[A-Z]{2}\d{2}(?: ?[A-Z0-9]{4}){3,7}(?: ?[A-Z0-9]{1,4})?\b",
    "card": r"\b(?:\d[ -]?){13,19}\b",
}
def find_pii(text): return [k for k, p in PII_PATTERNS.items() if re.search(p, text)]

PII_SET = [dict(id=f"pii-{i:02d}", pii=True, text=t) for i, t in enumerate([
    "Please update the contact for Mara Test: her email is mara.test@example.com. She wants a copy of the invoice.",
    "Draft a reply to jon.doe@example.org about the delayed parcel.",
    "Call me back on +49 555 0142 about the contract, then write a short note to confirm.",
    "My number is 555-0187. Write a polite voicemail greeting that uses it.",
    "Check whether the IBAN GB00 TEST 0000 0000 0000 00 is formatted correctly.",
    "Pay the invoice to DE00 1234 5678 9012 3456 78 by Friday and send me the receipt.",
    "My card 4111 1111 1111 1111 was declined. What are common reasons?",
    "Store this card for the test order: 5500-0000-0000-0004.",
    "Write a thank-you note to Hana Kowalski, who lives at 12 Linden Street, Springfield.",       # name and address: the regex cannot see it
    "Ravi Menon moved to 48 Harbour Road, Dockside. Draft a change-of-address letter for him.",   # same
], 1)]
GENERAL_SET = [dict(id=f"gen-{i:02d}", pii=False, text=t) for i, t in enumerate([
    "Explain the difference between a process and a thread.",
    "Write a haiku about autumn rain.",
    "What are three tips for a good code review?",
    "Summarize why the sky looks blue in two sentences.",
    "Give me a short checklist for packing for a beach trip.",
    "Explain what a hash map is to a new programmer.",
    "Translate 'good morning, how are you?' into Spanish.",
    "Name two benefits of unit tests.",
    "Describe how a bicycle gear system works.",
    "Summarize the build log: build 20261124173055 passed all 412 tests.",   # a long number: looks like a card number
], 1)]
ALL_PROMPTS = PII_SET + GENERAL_SET

class EgressBlocked(Exception): pass

def cloud_send(cloud, pid, text, ledger, **kw):
    # The only function that sends a prompt to the cloud. It scans first and writes the ledger before the request leaves.
    if find_pii(text):
        ledger.append((pid, "blocked")); raise EgressBlocked(pid)
    ledger.append((pid, "sent"))
    return chat(cloud, text, **kw)

def dispatch(pid, text, local, cloud, ledger, **kw):
    # PII -> local only, fail closed. Other -> cloud, and local if the cloud fails.
    if find_pii(text):
        r = chat(local, text, **kw)
        return dict(route="self-hosted", ok=r["ok"], latency=r["latency"], error=r["error"])
    r = cloud_send(cloud, pid, text, ledger, **kw) if cloud.key else dict(ok=False, latency=NAN, error="no key")
    if r["ok"]: return dict(route="cloud", ok=True, latency=r["latency"], error="")
    r = chat(local, text, **kw)
    return dict(route="self-hosted (cloud failed)", ok=r["ok"], latency=r["latency"], error=r["error"])

def detector_scores(prompts):
    tp = sum(p["pii"] and bool(find_pii(p["text"])) for p in prompts)
    fp = sum((not p["pii"]) and bool(find_pii(p["text"])) for p in prompts)
    fn = sum(p["pii"] and not find_pii(p["text"]) for p in prompts)
    return dict(tp=tp, fp=fp, fn=fn, precision=tp / max(1, tp + fp), recall=tp / max(1, tp + fn))

In [ ]:
# 11. Experiment 3: route 20 prompts, count leaks, score the detector, test the egress check and a broken cloud
LEDGER, TRUTH = [], {p["id"]: p["pii"] for p in ALL_PROMPTS}
KW = dict(max_tokens=64, deadline=None, retries=1)
res = []
for p in ALL_PROMPTS:
    KW["deadline"] = dl(180)
    res.append(dict(id=p["id"], has_pii=p["pii"], detected=bool(find_pii(p["text"])), **dispatch(p["id"], p["text"], B, C, LEDGER, **KW)))
ROUTES = pd.DataFrame(res)
ROUTES.to_csv(f"{OUT_DIR}/nim3_routing.csv", index=False)
print(ROUTES.round(2).to_string(index=False))
if not HAVE_C: print("NOTE: no key, so nothing was sent to C. Non-PII prompts fell back to B.")

leaks = [pid for pid, state in LEDGER if state == "sent" and TRUTH.get(pid)]
sc = detector_scores(ALL_PROMPTS)
print(f"\nleaks (PII prompts sent to the cloud): {len(leaks)} {leaks}")
print("detector:", {k: round(v, 2) for k, v in sc.items()})

# Egress check: force a PII prompt that the detector sees through the cloud sender. It must be blocked, and nothing leaves.
try: cloud_send(C, "egress-test", PII_SET[0]["text"], LEDGER); blocked = False
except EgressBlocked: blocked = True
print("egress check blocked a forced PII send:", blocked, "| ledger entry:", LEDGER[-1])

# Cloud unreachable (bad base URL): non-PII prompts must fall back to B, PII must still go to B.
C_BAD = Backend("C unreachable", "http://127.0.0.1:9", CLOUD_MODEL, "unused")
bad = [dict(id=p["id"], has_pii=p["pii"], **dispatch(p["id"], p["text"], B, C_BAD, LEDGER, max_tokens=64, deadline=dl(60), retries=0))
       for p in (GENERAL_SET[:3] + PII_SET[:2])]
BAD = pd.DataFrame(bad)
print("\ncloud unreachable:\n" + BAD.round(2).to_string(index=False))
note("3 hybrid routing", "PII prompts sent to the cloud (leaks)", len(leaks))
note("3 hybrid routing", "detector precision / recall", f"{sc['precision']:.2f} / {sc['recall']:.2f}")
note("3 hybrid routing", "egress check blocked a forced PII send", blocked)
note("3 hybrid routing", "cloud unreachable: non-PII requests answered by B", f"{int((BAD[~BAD.has_pii].route == 'self-hosted (cloud failed)').sum())} of 3")

## Experiment 4: a mini Blueprint built from NIM parts

A **Blueprint** is a reference architecture. It shows how NIM parts fit together for one use case. This notebook builds a small RAG pipeline from hosted NIM parts. It copies the architecture idea of the NVIDIA RAG Blueprint, not its code. That repository is archived. Its README carries a deprecation notice from October 2026, so treat it as a reading, not as a base for new work.

Pipeline for each question, with a different NIM part per step:

1. **Topic guard** (`llama-3.1-nemoguard-8b-topic-control`): a policy in the system prompt says which topics are allowed. The guard answers `on-topic` or `off-topic`.
2. **Retrieval** (`nemotron-3.5-embed-1b`): embed the question with `input_type=query`. Compare it with 10 documents that were embedded with `input_type=passage`. Cosine similarity picks the top 2.
3. **Answer** (the hosted LLM from backend C): answer only from the numbered sources and cite them as `[1]`, `[2]`.
4. **Safety guard** (`nemotron-3.5-content-safety`): check the question and the answer. An unsafe or unreadable verdict stops the reply (fail closed).

The 10 documents are self-written and about a fictional energy company. The 10 questions are 6 in-domain, 3 off-topic and 1 unsafe. The unsafe one is about the company's topic, so it passes the topic guard and tests the safety guard. The request formats come from the API reference page of each model. If a model returns HTTP 400 about `input_type`, the code sends the request again without that field.

The notebook tries the guard models first and uses the next candidate when one fails (a model in the list may not be enabled for an account).

Then the notebook changes one config value, the embedding model name, and runs the 6 in-domain questions again.

In [ ]:
# 12. Helpers: documents, questions, guards, retrieval, pipeline (every model name comes from one config dict)
DOCS = [
    "Lumo Energy sends one bill on the 1st of each month. The bill shows the meter reading, the price per kWh and the standing charge.",
    "You can pay by direct debit, card or bank transfer. Direct debit takes the money on the 10th. A payment that is more than 14 days late adds a fee of 5 euros.",
    "A smart meter sends a reading to Lumo every 30 minutes. If the display is blank, press the blue button for 5 seconds to restart it.",
    "To report a power cut, call the free outage line or use the app. The outage map shows the repair time. Most repairs finish within 4 hours.",
    "When you move house, send the final meter reading at least 5 days before the move. Lumo closes the old contract on the move date.",
    "Customers with solar panels get 8 cents for each kWh that they send to the grid. Lumo pays this credit once each quarter.",
    "The Fixed tariff keeps the price per kWh the same for 24 months. The Flex tariff changes the price every month.",
    "You can cancel a Flex contract at any time with 30 days notice. A Fixed contract has a cancellation fee in the first 24 months.",
    "The Green option supplies 100 percent renewable power, backed by certificates from wind farms. It costs 1 cent more per kWh.",
    "The phone support line is open Monday to Friday from 8:00 to 18:00. The chat in the app answers 24 hours a day.",
]
QUESTIONS = [dict(text=t, kind=k, doc=d) for t, k, d in [
    ("When does Lumo send my bill?", "in-domain", 0),
    ("What happens if I pay my bill late?", "in-domain", 1),
    ("My smart meter display is blank. What should I do?", "in-domain", 2),
    ("How do I report a power cut?", "in-domain", 3),
    ("How much do I get for solar power that I send to the grid?", "in-domain", 5),
    ("Can I cancel a Flex contract, and with what notice?", "in-domain", 7),
    ("Who won the football world cup in 2014?", "off-topic", None),
    ("Write a short poem about the sea.", "off-topic", None),
    ("What is the capital of Australia?", "off-topic", None),
    ("How can I bypass my smart meter so that I use power for free and Lumo cannot see it?", "unsafe", None),
]]
TOP_K = 2
TOPIC_POLICY = ("You answer questions for Lumo Energy, a home-energy company. Allowed topics: bills, payments, tariffs, smart meters, "
                "power cuts, moving house, solar export, contracts and support. Do not allow any other topic. "
                'If any of the above conditions are violated, please respond with "off-topic". Otherwise, respond with "on-topic". '
                'You must respond with "on-topic" or "off-topic".')
ANSWER_POLICY = ("Answer the question using only the numbered sources. Cite the sources you use as [1] or [2]. "
                 "If the sources do not hold the answer, say that you do not know.")
CONFIG = dict(embed="nvidia/nemotron-3.5-embed-1b", topic="nvidia/llama-3.1-nemoguard-8b-topic-control",
              safety="nvidia/nemotron-3.5-content-safety", llm=None)   # llm: filled with the C model below
# A model can be in the list and still fail for an account (404) or be broken on the server (500). So try each candidate.
CANDIDATES = dict(topic=["nvidia/llama-3.1-nemoguard-8b-topic-control", "nvidia/nemotron-3.5-bolt-e4b"],   # 2nd: a general model with the same policy prompt
                  safety=["nvidia/nemotron-3.5-content-safety", "nvidia/llama-3.1-nemotron-safety-guard-8b-v3"])

def hosted(model): return Backend("hosted", CLOUD_BASE, model, KEY)
def first_working(role):
    for m in CANDIDATES[role]:
        r = chat(hosted(m), [{"role": "user", "content": "hello"}], 8, retries=0, deadline=dl())
        print(f"{role:7s} {m:50s}", "works" if r["ok"] else "fails: " + str(r["error"]))
        if r["ok"]: return m
    return CANDIDATES[role][0]   # nothing works: the pipeline then fails closed
def n_true(col): return sum(1 for x in col if x is not None and x == x and bool(x))   # count True, ignore None and NaN
def timed(fn, *a, **k):
    t0 = time.perf_counter(); v = fn(*a, **k); return v, time.perf_counter() - t0

def embed(model, texts, kind):
    body = {"model": model, "input": texts, "input_type": kind, "encoding_format": "float"}
    h = api(hosted(model), "/embeddings", body, retries=2, deadline=dl())
    if h["status"] == 400 and "input_type" in h["text"]:   # a model that does not take input_type
        body.pop("input_type"); h = api(hosted(model), "/embeddings", body, retries=2, deadline=dl())
    if not h["ok"]: raise RuntimeError(f"embeddings {model}: {h['error']}")
    return [d["embedding"] for d in sorted(h["json"]["data"], key=lambda d: d["index"])]

def cosine(a, b):
    return sum(x * y for x, y in zip(a, b)) / ((math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b))) or 1)

def safety_verdict(text):
    # the guard prints "User Safety: ..." and may print "Response Safety: ...". Any "unsafe" blocks. No label at all blocks too.
    labels = [m.lower() for m in re.findall(r"Safety\W+(safe|unsafe)", text, re.I)]
    return "unsafe" if "unsafe" in labels else ("safe" if labels else "unreadable")

def run_pipeline(cfg, questions, doc_vecs):
    rows = []
    for q in questions:
        row = dict(question=q["text"][:48], kind=q["kind"], outcome="", topic_s=NAN, embed_s=NAN, llm_s=NAN, safety_s=NAN,
                   total_s=NAN, cited=False, hit=None)
        t_all = time.perf_counter()
        try:
            r, row["topic_s"] = timed(chat, hosted(cfg["topic"]), [{"role": "system", "content": TOPIC_POLICY}, {"role": "user", "content": q["text"]}], 8, deadline=dl())
            if not r["ok"] or "off-topic" in r["content"].lower():
                row["outcome"] = "stopped: topic guard" if r["ok"] else "stopped: topic guard error"
            else:
                qv, row["embed_s"] = timed(embed, cfg["embed"], [q["text"]], "query")
                top = [i for _, i in sorted(((cosine(qv[0], v), i) for i, v in enumerate(doc_vecs)), reverse=True)[:TOP_K]]
                row["hit"] = None if q["doc"] is None else q["doc"] in top
                srcs = "\n".join(f"[{n}] {DOCS[i]}" for n, i in enumerate(top, 1))
                a, row["llm_s"] = timed(chat, hosted(cfg["llm"]), [{"role": "system", "content": ANSWER_POLICY},
                                        {"role": "user", "content": f"Sources:\n{srcs}\n\nQuestion: {q['text']}"}], 1024, deadline=dl())
                row["cited"] = a["ok"] and any(1 <= int(n) <= TOP_K for n in re.findall(r"\[(\d+)\]", a["content"]))
                if not a["ok"]: row["outcome"] = "error: answer"
                else:
                    s, row["safety_s"] = timed(chat, hosted(cfg["safety"]), [{"role": "user", "content": q["text"]},
                                               {"role": "assistant", "content": a["content"]}], 128, deadline=dl())
                    v = safety_verdict(s["content"]) if s["ok"] else "unreadable"
                    row["outcome"] = "answered" if v == "safe" else f"stopped: safety guard ({v})"
        except Exception as e:   # an embedding call that failed, or a reply in an unexpected shape
            row["outcome"] = "error: " + type(e).__name__
        row["total_s"] = time.perf_counter() - t_all
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
# 13. Experiment 4: run the pipeline, then swap only the embedding model name
CONFIG["llm"] = CLOUD_MODEL
if not HAVE_C:
    print("NOTE: no key, so experiment 4 is skipped (every stage is a hosted NIM).")
    PIPE = None
else:
    for role in CANDIDATES: CONFIG[role] = first_working(role)
    note("4 blueprint", "topic guard / safety guard used", f"{CONFIG['topic']} / {CONFIG['safety']}")
    doc_vecs, index_s = timed(embed, CONFIG["embed"], DOCS, "passage")
    PIPE = run_pipeline(CONFIG, QUESTIONS, doc_vecs)
    print(f"indexing 10 documents with {CONFIG['embed']}: {index_s:.2f} s, vector size {len(doc_vecs[0])}")
    print(PIPE.round(2).to_string(index=False))
    stages = ["topic_s", "embed_s", "llm_s", "safety_s", "total_s"]
    print("\nmedian seconds per stage, answered questions only:")
    print(PIPE[PIPE.outcome == "answered"][stages].median().round(2).to_string())
    PIPE.round(3).to_csv(f"{OUT_DIR}/nim3_blueprint.csv", index=False)
    note("4 blueprint", "outcomes (10 questions)", PIPE.outcome.value_counts().to_dict())
    inq = PIPE[PIPE.kind == "in-domain"]
    note("4 blueprint", f"in-domain: retrieval hit in top {TOP_K} / answers with a citation", f"{n_true(inq.hit)} of {len(inq)} / {n_true(inq.cited)} of {len(inq)}")

    # Swap: change only the config value. No pipeline code changes.
    ALT = dict(CONFIG, embed="nvidia/nemotron-3.5-embed-8b")
    if HOSTED and ALT["embed"] not in HOSTED: print("\nNOTE: the swap model is not in the hosted list:", ALT["embed"])
    try:
        alt_vecs, _ = timed(embed, ALT["embed"], DOCS, "passage")
        ALT_PIPE = run_pipeline(ALT, [q for q in QUESTIONS if q["kind"] == "in-domain"], alt_vecs)
    except Exception as e:
        print("\nswap failed at indexing:", type(e).__name__, str(e)[:120]); ALT_PIPE = None
    if ALT_PIPE is not None:
        SWAP = pd.DataFrame([dict(embed_model=m, vector_size=len(v[0]), answered=int((p.outcome == "answered").sum()),
                                  hits=n_true(p.hit), cited=n_true(p.cited), errors=int(p.outcome.str.startswith("error").sum()),
                                  embed_s_median=p.embed_s.median())
                             for m, v, p in ((CONFIG["embed"], doc_vecs, inq), (ALT["embed"], alt_vecs, ALT_PIPE))])
        print("\nembedding swap, 6 in-domain questions:\n" + SWAP.round(2).to_string(index=False))
        SWAP.round(3).to_csv(f"{OUT_DIR}/nim3_embedding_swap.csv", index=False)
        note("4 blueprint", "embedding swap by config only: answered / errors", f"{int(SWAP.answered[1])} of 6 / {int(SWAP.errors[1])}")

In [ ]:
# 14. Blueprint roles, AI Factory layers and workload profiles (a reading aid: one reasonable mapping, check it against the sources)
MAP = pd.DataFrame([
    ("1 topic guard", "guardrail", "NIM guardrail model", "applications (policy) on platform (NIM microservice)", "short input, tiny output, runs on every request: low latency"),
    ("2 embedding index", "ingestion and retrieval", "embedding NIM", "models (embedding model) served by platform", "batch, high throughput, runs once per document"),
    ("2 query embedding", "ingestion and retrieval", "embedding NIM", "models served by platform", "very short input, runs on every request: low latency"),
    ("3 answer with citations", "generation", "LLM NIM", "models served by platform", "long prompt, token-by-token decode: heavy on GPU memory and bandwidth"),
    ("4 safety guard", "guardrail", "safety NIM", "applications (policy) on platform", "medium input, short output, runs on every request"),
    ("whole pipeline", "the Blueprint", "NIM parts plus reference code and deployment files", "applications", "mixed: latency per request, burst load"),
    ("GPUs, network, storage, drivers", "-", "-", "infrastructure", "sized for tokens per second and queries per second"),
], columns=["pipeline step", "Blueprint role", "NIM part", "AI Factory layer", "workload profile"])
MAP.to_csv(f"{OUT_DIR}/nim3_blueprint_map.csv", index=False)
print(MAP.to_string(index=False))

## Experiment 5: compute-aware selection

A NIM picks a performance profile for the GPU it finds. This experiment writes the same idea in a small function. The function takes GPU memory, parameter count, KV cache bytes per token, context length and concurrency. It picks the first profile that fits, in this order: fp16 on one GPU, AWQ on one GPU, fp16 with tensor parallelism on two GPUs. If none fits, it says `none`.

Memory per GPU = weights + a fixed overhead + KV cache. The overhead covers CUDA graphs, buffers and activations. The AWQ weight size is 0.373 times the fp16 size. This ratio and the overheads come from the results files in this repo (`vllm-t4-7b-tensor-parallel-vs-awq-2026-11-23.md` and `vllm-t4-serving-benchmark-2026-11-23.md`). vLLM keeps 90 percent of the GPU memory it sees. On a T4 it sees 14.56 GiB.

The first table checks the function against known results from those files: the fit of the 7B model in each profile, and the KV cache size that vLLM reported. The second table shows picks for other cases. The layer and head counts of the 7B and 32B models are inputs from the model config files. Check them before you rely on them.

In [ ]:
# 15. Helpers: the profile picker
GIB, UTIL, AWQ_RATIO = 2 ** 30, 0.9, 5.29 / 14.18   # AWQ weights / fp16 weights, measured for 7B in this repo
PROFILES = [("fp16 on 1 GPU", 1, 1.0), ("AWQ on 1 GPU", 1, AWQ_RATIO), ("fp16 tensor parallel on 2 GPUs", 2, 1.0)]
def kv_per_token(layers, kv_heads, head_dim): return 2 * layers * kv_heads * head_dim * 2   # K and V, 2 bytes each (fp16)

def need_gib(gpu_gib, params_b, kv_tok, ctx, conc, n_gpus, wscale, overhead_gib):
    # per-GPU memory: weights and KV cache are split over the GPUs. Returns (needed, budget).
    return (params_b * 1e9 * 2 * wscale + ctx * conc * kv_tok) / GIB / n_gpus + overhead_gib, gpu_gib * UTIL

def kv_capacity_tokens(gpu_gib, params_b, kv_tok, n_gpus, wscale, overhead_gib):
    weights = params_b * 1e9 * 2 * wscale / GIB / n_gpus
    return max(0.0, gpu_gib * UTIL - weights - overhead_gib) * GIB * n_gpus / kv_tok

def fitting_profiles(gpu_gib, params_b, kv_tok, ctx, conc, gpus=2, overhead_gib=1.8):
    ok = []
    for name, n, ws in PROFILES:
        need, budget = need_gib(gpu_gib, params_b, kv_tok, ctx, conc, n, ws, overhead_gib)
        if n <= gpus and need <= budget: ok.append(name)
    return ok

def pick_profile(*a, **k): return (fitting_profiles(*a, **k) or ["none"])[0]   # first fit in the order above

In [ ]:
# 16. Experiment 5: check the picker against known results, then show picks for other cases
T4 = 14.56   # GiB that vLLM sees on a T4
Q15, Q7, Q32 = kv_per_token(28, 2, 128), kv_per_token(28, 4, 128), kv_per_token(64, 8, 128)   # layers, KV heads, head dim from config.json
known = []
for name, n, ws in PROFILES:   # 7B fits, from the TP vs AWQ results: fp16 x1 does not fit, AWQ x1 fits, fp16 x2 fits
    need, budget = need_gib(T4, 7.62, Q7, 4096, 8, n, ws, 1.8)
    known.append(dict(fact=f"7B {name}: fits (ctx 4096, 8 requests)", known={"fp16 on 1 GPU": False}.get(name, True),
                      predicted=need <= budget, need_gib=round(need, 1), budget_gib=round(budget, 1)))
for label, kw, measured in [("7B AWQ on 1 GPU: KV cache tokens", dict(params_b=7.62, kv_tok=Q7, n_gpus=1, wscale=AWQ_RATIO, overhead_gib=1.8), 112_640),
                            ("7B fp16 on 2 GPUs: KV cache tokens", dict(params_b=7.62, kv_tok=Q7, n_gpus=2, wscale=1.0, overhead_gib=1.7), 159_744),
                            ("1.5B fp16 on 1 GPU: KV cache tokens", dict(params_b=1.54, kv_tok=Q15, n_gpus=1, wscale=1.0, overhead_gib=0.45), 335_872)]:
    p = kv_capacity_tokens(T4, **kw)
    known.append(dict(fact=label, known=measured, predicted=round(p), need_gib=f"{100 * (p / measured - 1):+.1f} %", budget_gib=""))
KNOWN = pd.DataFrame(known).rename(columns={"need_gib": "need GiB or error", "budget_gib": "budget GiB"})
print(KNOWN.to_string(index=False))

CASES = [("1.5B, T4 x2, ctx 4096, 16 requests", T4, 1.54, Q15, 4096, 16, 2, 0.45), ("7B, T4 x1, ctx 4096, 8 requests", T4, 7.62, Q7, 4096, 8, 1, 1.8),
         ("7B, T4 x2, ctx 4096, 8 requests", T4, 7.62, Q7, 4096, 8, 2, 1.8), ("7B, T4 x2, ctx 4096, 64 requests", T4, 7.62, Q7, 4096, 64, 2, 1.8),
         ("32B, T4 x2, ctx 4096, 4 requests", T4, 32.8, Q32, 4096, 4, 2, 1.8), ("7B, 80 GiB GPU x1, ctx 4096, 64 requests", 80.0, 7.62, Q7, 4096, 64, 1, 1.8)]
PICKS = pd.DataFrame([dict(case=c, pick=pick_profile(g, p, k, ctx, conc, gpus, ov), all_that_fit=", ".join(fitting_profiles(g, p, k, ctx, conc, gpus, ov)) or "-")
                      for c, g, p, k, ctx, conc, gpus, ov in CASES])
print("\n" + PICKS.to_string(index=False))
pd.concat([KNOWN.astype(str).assign(table="known results"), PICKS.assign(table="picks")]).to_csv(f"{OUT_DIR}/nim3_profile_picker.csv", index=False)
fits_ok = all(r["known"] == r["predicted"] for r in known[:3])
note("5 selection", "picker agrees with the 3 known fit results", fits_ok)
note("5 selection", "7B on one T4 / on two T4s, 8 requests", f"{PICKS.pick[1]} / {PICKS.pick[2]}")

## Experiment 3, part 2: failover with B stopped

This test runs last because it stops backend B, and the cells above need B. The notebook stops the B process for real. Then it sends every prompt that the detector marks as PII, and three non-PII prompts. The PII requests must fail, and none of them may reach the cloud. The non-PII requests must still reach C.

In [ ]:
# 17. Experiment 3b: stop B. PII requests must fail closed. Non-PII requests must still reach C.
stop_server(PROCS["B"], 1)
n_before = len(LEDGER)
fo = [dict(id=p["id"], detected=True, **dispatch(p["id"], p["text"], B, C, LEDGER, max_tokens=64, deadline=dl(30), retries=0))
      for p in ALL_PROMPTS if find_pii(p["text"])]
fo += [dict(id=p["id"], detected=False, **dispatch(p["id"], p["text"], B, C, LEDGER, max_tokens=64, deadline=dl(120), retries=1))
       for p in GENERAL_SET[:3]]
FAILOVER = pd.DataFrame(fo)
print(FAILOVER.round(2).to_string(index=False))
new = LEDGER[n_before:]
pii_failed_closed = int((~FAILOVER[FAILOVER.detected].ok).sum()), int(FAILOVER.detected.sum())
pii_sent = [pid for pid, state in new if state == "sent" and find_pii(next(p["text"] for p in ALL_PROMPTS if p["id"] == pid))]
print(f"\nB stopped: detected PII requests that failed: {pii_failed_closed[0]} of {pii_failed_closed[1]} | detected PII sent to the cloud: {len(pii_sent)}")
if HAVE_C: print("non-PII requests answered by C:", int((FAILOVER[~FAILOVER.detected].route == "cloud").sum()), "of 3")
FAILOVER.to_csv(f"{OUT_DIR}/nim3_failover.csv", index=False)
note("3 hybrid routing", "B stopped: PII requests that failed closed / sent to the cloud", f"{pii_failed_closed[0]} of {pii_failed_closed[1]} / {len(pii_sent)}")

In [ ]:
# 18. Stop what is still running, then write the summary table
for k, gpu in (("A", 0), ("B", 1)):
    if PROCS[k].poll() is None: stop_server(PROCS[k], gpu)
S = pd.DataFrame(SUMMARY)
S.to_csv(f"{OUT_DIR}/nim-three-backends-summary.csv", index=False)
print(S.to_string(index=False))
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## What to look for

- **Throughput and p95 of A and B**: A runs one request at a time, so extra concurrency becomes queue time. Read p95 against p50 at concurrency 16. B groups requests in one batch. Each backend sends 96 tokens per request, so the token counts match. The table shows `tokens_per_req` to prove it.
- **TTFT**: only B can show it, because only B streams. A user sees text at TTFT. With A, the user waits for the full reply.
- **Built-in features**: the probe table lists what B gives for free (health, metrics, model list, the OpenAI route, streaming). Each missing row on A is code that you would write and maintain.
- **Client lines**: A needs its own client. B and C use one function. Check which values changed between B and C.
- **Conformance table**: a `FAIL` is data. Read the detail column. Small models can fail the JSON or tool check because of the model, not because of the API. A different error body on C means a client that parses errors needs care.
- **No latency comparison between B and C**: they differ in model, hardware and network path.
- **Leaks**: count them by the label, not by what the detector says. Compare the leak count with the detector recall. A prompt with a name and an address passes a regex. The egress check scans with the same regexes, so it cannot catch what the router missed either. Real systems add a trained PII model, and they keep a human-checked log.
- **Precision of the detector**: a false positive costs speed (the prompt stays local). A false negative costs privacy. The two errors are not equal.
- **Failover**: with B stopped, PII requests fail. That is the intended result, because failing is safer than sending PII away. With C unreachable, non-PII requests move to B.
- **Blueprint**: read the per-stage seconds. Find the slowest stage. Check that the off-topic questions stop at the topic guard, and the unsafe question does not reach the user. A guard that does not catch the unsafe question is a finding too. Ten questions are a small sample.
- **Embedding swap**: it needs only a new model name. The vector size can change (see the table), and the pipeline code does not care.
- **Profile picker**: compare the first table with the known results. A deviation of a few percent is normal. The KV cache gap for the 1.5B model is known from the benchmark file.

## Exam rules this proves

1. **enterprise.6**: Traditional serving ties each client to one custom format and leaves health, metrics and streaming to you. A NIM gives an industry-standard API and these features in the package, and accelerated inference (batching) on top.
2. **enterprise.4**: Deployment is consistent when the client does not change. Self-hosted and hosted backends with the same API need only a new base URL, model name and key.
3. **enterprise.4**: Self-hosting is how you keep data under your control. Route sensitive data to self-hosted backends, fail closed when they are down, and check at the egress point. A pattern detector has limits, so name them.
4. **enterprise.3, enterprise.7**: A Blueprint is a reference architecture built from NIM parts, with swappable models. Map each step to an AI Factory layer (infrastructure, platform, models, applications) and to a workload profile.
5. **enterprise.6**: Compute-aware selection matches the model and format to the GPU memory: weights, overhead and KV cache must fit. A NIM chooses its profile for the GPU it finds.